# Llama-3.2-1B × OpenMathInstruct-2 leaderboard — robustness (9000 steps)

Cross-**model** *and* cross-**dataset** robustness in one cell: does `adam-polar-product-lora-coupled-spectral-chord-tight` (ns=8, full Newton–Schulz whitening, picard=1) keep its eval-loss edge over AdamW-LoRA when the base is **Llama-3.2-1B** (not OLMo) *and* the task is **math-IFT** (OpenMathInstruct-2, not code)?

Setup: Llama-3.2-1B × OpenMathInstruct-2 `train_2M` (Llama-tokenized cache `data/openmath_instruct_2_2m_packed_seq2048_llama32`, 512,440 packed slots @ seq=2048) × global_batch=16 (4×4) × packed_v1.1 × constant LR × α=r × all-linear × bf16 × compile × single-GPU Blackwell. `max_steps=9000`, `eval_every=250`.

- **AdamW**: η ∈ {3e-5, 1e-4, 3e-4, 1e-3}
- **chord-tight ns=8** (`adam-polar-product-lora-coupled-spectral-chord-tight`, `--muon_ns_steps 8 --polar_method ns`, picard=1): η ∈ {3e-3, 1e-2, 3e-2, 1e-1} — grid pre-extended to 1e-1 because OLMo×math's ns=8 pinned at the 1e-2 grid edge.

Source log groups: `adamw_robustness_llama32_1b_openmath_r{64,256}_blackwell` and `chord_tight_robustness_llama32_1b_openmath_r{64,256}_ns8_blackwell`. No ns=5 arm here (OLMo×math established ns=8 ≈ ns=5; this notebook goes straight to ns=8). Grids are single-sweep — no separate `_ext_right` groups.

**σ anchor**: no per-(model,dataset) multi-seed AdamW run. Δ quoted against `σ_AdamW(packed_v1, opc-sft-stage2, r=64) = 0.0017` as a **loose proxy only** — now cross-model *and* cross-dataset, so treat σ-units as indicative, not rigorous, until re-anchored.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
import matplotlib.pyplot as plt

# Repo root by marker-walk, so this notebook works from any subdir.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'lora_playground').is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from lora_playground.plotting import leaderboard_panel, canonical_label

# Run membership comes from the shared registry (lora_playground.workloads) — the
# SAME source the leaderboard doc uses, so notebook and doc cannot drift. Cells are
# leaderboard_panel(model, dataset, rank, ...) + an optional label_filter(label, cfg).

def ns_of(cfg):
    oc = cfg.get('optimizer_config') or {}
    return cfg.get('muon_ns_steps', oc.get('ns_steps'))

def picard_of(cfg):
    return cfg.get('_derived', {}).get('effective_picard_iters', cfg.get('picard_iters_override')) or 1

def is_curv(label):
    # curvature-whitening / SOAP-curv / KL-Shampoo arms.
    return ('SOAP-curv' in label) or ('KL-Shampoo' in label) or ('+curv' in label)

## r=64

In [ ]:
_fig, _tdf, sdf = leaderboard_panel('meta-llama/Llama-3.2-1B', 'openmath', 64,
    'Llama-3.2-1B × OpenMathInstruct-2 r=64',
    figsize=(11, 4))
plt.show()
sdf

## r=256

In [ ]:
_fig, _tdf, sdf = leaderboard_panel('meta-llama/Llama-3.2-1B', 'openmath', 256,
    'Llama-3.2-1B × OpenMathInstruct-2 r=256',
    figsize=(11, 4))
plt.show()
sdf